# Notebook 04 — Scorecard y Análisis de Variables de Riesgo

**Entrada:** `splits.pkl`, `modelo_final.pt`, `loan_procesado.parquet`, `probabilidades_test.csv`  
**Salida:** `scorecard.csv`, gráficas de la scorecard y análisis de riesgo

## Contenido
1. Configuración del entorno
2. Carga de datos y modelo
3. Construcción de la scorecard
4. Calibración de puntajes
5. Distribución de scores por clase
6. Análisis de variables de riesgo
7. Segmentación por bandas de riesgo
8. Caso de uso: perfil de ejemplo

## 1. Configuración del entorno

In [ ]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

import scorecardpy as sc

from sklearn.metrics import roc_auc_score, roc_curve

# Configuracion
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', font_scale=1.1)

# Rutas
DATA_PATH    = os.path.join('..', 'data')
FIGURES_PATH = os.path.join('..', 'figures')
OUTPUT_PATH  = os.path.join('..', 'outputs')
MODELS_PATH  = os.path.join('..', 'models')

print('Entorno configurado.')

## 2. Carga de datos y modelo

Cargamos los componentes necesarios de los notebooks anteriores:
- El dataset procesado (antes de WOE) para construir la scorecard con scorecardpy.
- Los splits para evaluar sobre el conjunto de test.
- Las probabilidades predichas por el mejor modelo.

In [ ]:
# Dataset procesado (con las variables originales, antes de WOE)
df_procesado = pd.read_parquet(os.path.join(DATA_PATH, 'loan_procesado.parquet'))
print(f'Dataset procesado: {df_procesado.shape}')

# Splits
splits = joblib.load(os.path.join(OUTPUT_PATH, 'splits.pkl'))
X_train = splits['X_train']
y_train = splits['y_train']
X_test  = splits['X_test']
y_test  = splits['y_test']

# Probabilidades del modelo final (generadas en Notebook 03)
df_probas = pd.read_csv(os.path.join(OUTPUT_PATH, 'probabilidades_test.csv'))
print(f'Probabilidades cargadas: {df_probas.shape}')
print(f'\nDistribucion de probabilidades predichas:')
print(df_probas['proba_default'].describe())

## 3. Construcción de la scorecard

### Metodología

Una **scorecard** convierte las probabilidades de incumplimiento en un puntaje entero
interpretable, siguiendo la metodología estándar de la industria de riesgo de crédito
(Siddiqi, 2006).

La conversión se basa en la relación lineal entre el score y el log-odds:

$$\text{Score} = \text{Offset} + \text{Factor} \times \ln\left(\frac{1-p}{p}\right)$$

Donde:
- $p$ es la probabilidad de incumplimiento
- **Factor** = PDO / ln(2), con PDO (Points to Double the Odds)
- **Offset** = Score objetivo - Factor × ln(Odds objetivo)

### Parámetros de calibración

Se utilizan los parámetros convencionales de la industria:
- **Score base:** 600 puntos (para un odds de 50:1, es decir, probabilidad de ~2%)
- **PDO (Points to Double Odds):** 20 puntos

Esto significa que por cada 20 puntos adicionales de score, la probabilidad
de incumplimiento se reduce a la mitad.

In [ ]:
# --- Parametros de la scorecard ---
SCORE_BASE   = 600    # Score asignado a un odds de referencia
ODDS_BASE    = 50     # Odds de referencia (50:1 = ~2% de probabilidad)
PDO          = 20     # Points to Double the Odds

# Calculo de Factor y Offset
FACTOR = PDO / np.log(2)
OFFSET = SCORE_BASE - FACTOR * np.log(ODDS_BASE)

print(f'Parametros de la scorecard:')
print(f'  Score base : {SCORE_BASE} puntos (para odds {ODDS_BASE}:1)')
print(f'  PDO        : {PDO} puntos')
print(f'  Factor     : {FACTOR:.4f}')
print(f'  Offset     : {OFFSET:.4f}')

In [ ]:
def probabilidad_a_score(prob, offset=OFFSET, factor=FACTOR):
    """
    Convierte una probabilidad de incumplimiento a un puntaje de scorecard.
    
    Parametros:
        prob   : probabilidad de default (entre 0 y 1)
        offset : intercepto de la scorecard
        factor : pendiente de la scorecard
    
    Retorna:
        score  : puntaje entero
    """
    # Evitar log(0) o division por 0
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    odds = (1 - prob) / prob
    score = offset + factor * np.log(odds)
    return np.round(score).astype(int)


def score_a_probabilidad(score, offset=OFFSET, factor=FACTOR):
    """Convierte un score a probabilidad de incumplimiento."""
    odds = np.exp((score - offset) / factor)
    return 1 / (1 + odds)


# Aplicar la conversion al conjunto de test
df_probas['score'] = probabilidad_a_score(df_probas['proba_default'].values)

print('Conversion de probabilidades a scores:')
print(df_probas[['y_test', 'proba_default', 'score']].describe())

## 4. Verificación de la calibración

Se verifica que la relación entre el score y la probabilidad de incumplimiento
sea monótona y coherente.

In [ ]:
# Tabla de referencia: score vs probabilidad
scores_ref = np.arange(300, 850, 25)
probas_ref = score_a_probabilidad(scores_ref)

tabla_referencia = pd.DataFrame({
    'Score': scores_ref,
    'Probabilidad de Default (%)': (probas_ref * 100).round(2),
    'Odds (buenos:malos)': np.round((1 - probas_ref) / probas_ref, 1)
})

print('Tabla de referencia — Score vs Probabilidad de Incumplimiento:')
print(tabla_referencia.to_string(index=False))

In [ ]:
# Grafica de calibracion
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(scores_ref, probas_ref * 100, color='#F44336', linewidth=2.5,
         label='Probabilidad de default')
ax1.set_xlabel('Score', fontsize=12)
ax1.set_ylabel('Probabilidad de Default (%)', fontsize=12, color='#F44336')
ax1.tick_params(axis='y', labelcolor='#F44336')
ax1.set_ylim(0, 100)

ax2 = ax1.twinx()
ax2.plot(scores_ref, (1 - probas_ref) / probas_ref, color='#2196F3',
         linewidth=2.5, linestyle='--', label='Odds (buenos:malos)')
ax2.set_ylabel('Odds (buenos:malos)', fontsize=12, color='#2196F3')
ax2.tick_params(axis='y', labelcolor='#2196F3')

# Marcar el score base
ax1.axvline(SCORE_BASE, color='gray', linestyle=':', alpha=0.7)
ax1.annotate(f'Score base = {SCORE_BASE}',
             xy=(SCORE_BASE, 5), fontsize=10, ha='center',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

fig.suptitle('Calibracion de la Scorecard', fontsize=14, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.92), fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '11_calibracion_scorecard.png'), bbox_inches='tight')
plt.show()

## 5. Distribución de scores por clase

Se visualiza la separación entre buenos y malos pagadores en la escala de score.
Una buena scorecard debería mostrar distribuciones bien separadas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribucion de scores por clase', fontsize=14, fontweight='bold')

# Histograma
for clase, color, label in [(0, '#2196F3', 'Buen pagador'),
                             (1, '#F44336', 'Mal pagador')]:
    datos = df_probas.loc[df_probas['y_test'] == clase, 'score']
    axes[0].hist(datos, bins=40, alpha=0.6, color=color, label=label,
                 density=True, edgecolor='none')

axes[0].set_xlabel('Score')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Histograma de scores')
axes[0].legend()
axes[0].axvline(df_probas['score'].median(), color='gray', linestyle='--',
                alpha=0.7, label='Mediana global')

# KDE
sns.kdeplot(data=df_probas, x='score', hue='y_test', fill=True,
            common_norm=False, palette={0: '#2196F3', 1: '#F44336'},
            ax=axes[1])
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Densidad estimada (KDE)')
axes[1].legend(title='Clase', labels=['Buen pagador', 'Mal pagador'])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '12_distribucion_scores.png'), bbox_inches='tight')
plt.show()

# Estadisticas por clase
print('Estadisticas de score por clase:')
stats_clase = df_probas.groupby('y_test')['score'].describe()
stats_clase.index = ['Buen pagador', 'Mal pagador']
print(stats_clase)

In [ ]:
# Boxplot comparativo
fig, ax = plt.subplots(figsize=(8, 5))

bp = ax.boxplot(
    [df_probas.loc[df_probas['y_test'] == 0, 'score'],
     df_probas.loc[df_probas['y_test'] == 1, 'score']],
    patch_artist=True, notch=True, showfliers=True,
    flierprops=dict(marker='o', markersize=3, alpha=0.3),
    medianprops=dict(color='black', linewidth=2)
)

for patch, color in zip(bp['boxes'], ['#2196F3', '#F44336']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(['Buen pagador (0)', 'Mal pagador (1)'], fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Distribucion de scores por clase', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '13_boxplot_scores.png'), bbox_inches='tight')
plt.show()

## 6. Análisis de variables de riesgo

### ¿Qué variables hacen más riesgosa a una persona?

Para responder esta pregunta se utilizan dos enfoques complementarios:

1. **Information Value (IV):** indica el poder predictivo global de cada variable.
2. **Análisis WOE por categoría:** permite entender *cómo* y *en qué dirección* 
   cada valor de la variable afecta el riesgo.

Un valor WOE negativo indica una categoría asociada a mayor riesgo de incumplimiento,
mientras que un WOE positivo indica menor riesgo.

In [ ]:
# Recalcular bins WOE sobre el dataset procesado
bins = sc.woebin(df_procesado, y='default', check_cate_num=False, print_step=0)

# Extraer IV por variable
iv_df = pd.DataFrame([
    {'Variable': var, 'IV': bin_df['total_iv'].iloc[0]}
    for var, bin_df in bins.items()
]).sort_values('IV', ascending=False)

print('Information Value por variable (ordenado):')
print(iv_df.to_string(index=False))

In [ ]:
# Grafico de importancia de variables (IV)
fig, ax = plt.subplots(figsize=(10, max(6, len(iv_df) * 0.4)))

iv_plot = iv_df.sort_values('IV', ascending=True)

colores = [
    '#F44336' if v > 0.3  else
    '#FF9800' if v > 0.1  else
    '#4CAF50' if v > 0.02 else
    '#9E9E9E'
    for v in iv_plot['IV']
]

bars = ax.barh(iv_plot['Variable'], iv_plot['IV'],
               color=colores, edgecolor='white')

for bar in bars:
    if bar.get_width() > 0.01:
        ax.text(bar.get_width() + 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{bar.get_width():.3f}',
                va='center', fontsize=9)

# Umbrales de referencia
for umbral, etiqueta, color in [
    (0.02, 'Debil (<0.02)',   '#9E9E9E'),
    (0.1,  'Medio (0.1)',     '#4CAF50'),
    (0.3,  'Fuerte (0.3)',    '#F44336')
]:
    ax.axvline(umbral, linestyle='--', linewidth=1, color=color, alpha=0.6, label=etiqueta)

ax.set_xlabel('Information Value')
ax.set_title('Importancia de variables — Information Value', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '14_importancia_variables_iv.png'), bbox_inches='tight')
plt.show()

### 6.1 Análisis WOE de las variables más importantes

Para las variables con mayor IV, se analiza la estructura de riesgo por categoría.
Esto muestra concretamente qué valores de cada variable aumentan o disminuyen
la probabilidad de incumplimiento.

In [ ]:
# Seleccionar las top 8 variables por IV
top_vars_iv = iv_df.head(8)['Variable'].tolist()

print(f'Variables seleccionadas para analisis detallado: {top_vars_iv}')

In [ ]:
# Visualizacion WOE de las variables mas importantes
n_vars = len(top_vars_iv)
n_cols = 2
n_rows = -(-n_vars // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4.5))
axes = axes.flatten()

for i, var in enumerate(top_vars_iv):
    ax = axes[i]
    bin_data = bins[var].copy()
    
    # Obtener las categorias/rangos y sus WOE
    categorias = bin_data['breaks'].astype(str).values
    woe_vals   = bin_data['woe'].values
    count_vals = bin_data['count'].values
    bad_rate   = bin_data['badprob'].values
    
    # Colores segun signo del WOE
    colores = ['#F44336' if w < 0 else '#4CAF50' for w in woe_vals]
    
    bars = ax.bar(range(len(categorias)), woe_vals, color=colores,
                  edgecolor='white', alpha=0.85)
    
    # Etiquetas
    for j, (bar, br) in enumerate(zip(bars, bad_rate)):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01 * (1 if bar.get_height() >= 0 else -1),
                f'{br*100:.1f}%',
                ha='center', va='bottom' if bar.get_height() >= 0 else 'top',
                fontsize=8, fontweight='bold')
    
    ax.set_xticks(range(len(categorias)))
    ax.set_xticklabels(categorias, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('WOE')
    ax.set_title(f'{var} (IV = {iv_df[iv_df["Variable"]==var]["IV"].values[0]:.3f})',
                 fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

# Ocultar ejes sobrantes
for j in range(n_vars, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Analisis WOE de las variables mas importantes\n'
             '(rojo = mayor riesgo, verde = menor riesgo, etiqueta = tasa de default)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '15_analisis_woe_top_variables.png'), bbox_inches='tight')
plt.show()

### 6.2 Resumen de factores de riesgo

A partir del análisis WOE y IV, se construye una tabla resumen que describe
qué características hacen más riesgoso a un prestatario.

In [ ]:
# Construir tabla de factores de riesgo
factores_riesgo = []

for var in top_vars_iv:
    bin_data = bins[var]
    iv_val = iv_df[iv_df['Variable'] == var]['IV'].values[0]
    
    # Categoria de mayor riesgo (menor WOE)
    idx_max_riesgo = bin_data['woe'].idxmin()
    cat_riesgo     = bin_data.loc[idx_max_riesgo, 'breaks']
    tasa_riesgo    = bin_data.loc[idx_max_riesgo, 'badprob']
    
    # Categoria de menor riesgo (mayor WOE)
    idx_min_riesgo = bin_data['woe'].idxmax()
    cat_seguro     = bin_data.loc[idx_min_riesgo, 'breaks']
    tasa_seguro    = bin_data.loc[idx_min_riesgo, 'badprob']
    
    factores_riesgo.append({
        'Variable': var,
        'IV': round(iv_val, 3),
        'Categoria mas riesgosa': str(cat_riesgo),
        'Tasa default (riesgosa)': f'{tasa_riesgo*100:.1f}%',
        'Categoria mas segura': str(cat_seguro),
        'Tasa default (segura)': f'{tasa_seguro*100:.1f}%',
    })

df_factores = pd.DataFrame(factores_riesgo)
print('Factores de riesgo — Variables mas influyentes:')
print(df_factores.to_string(index=False))

## 7. Segmentación por bandas de riesgo

Se crean bandas de riesgo basadas en el score para facilitar la toma de decisiones.
Las bandas permiten clasificar a los solicitantes en categorías de riesgo discretas.

In [ ]:
# Definicion de bandas de riesgo
def asignar_banda(score):
    """Asigna una banda de riesgo basada en el score."""
    if score >= 700:
        return 'A - Riesgo muy bajo'
    elif score >= 650:
        return 'B - Riesgo bajo'
    elif score >= 600:
        return 'C - Riesgo medio'
    elif score >= 550:
        return 'D - Riesgo alto'
    else:
        return 'E - Riesgo muy alto'


df_probas['banda'] = df_probas['score'].apply(asignar_banda)

# Tabla resumen por banda
resumen_bandas = (
    df_probas.groupby('banda')
    .agg(
        n_total       = ('score', 'count'),
        score_min     = ('score', 'min'),
        score_max     = ('score', 'max'),
        score_medio   = ('score', 'mean'),
        tasa_default  = ('y_test', 'mean'),
        prob_media    = ('proba_default', 'mean')
    )
    .sort_index()
)

resumen_bandas['pct_total'] = (resumen_bandas['n_total'] / resumen_bandas['n_total'].sum() * 100)
resumen_bandas['tasa_default'] = resumen_bandas['tasa_default'] * 100
resumen_bandas['prob_media'] = resumen_bandas['prob_media'] * 100

print('Segmentacion por bandas de riesgo:')
print(resumen_bandas.round(2).to_string())

In [ ]:
# Visualizacion de bandas de riesgo
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Segmentacion por bandas de riesgo', fontsize=14, fontweight='bold')

colores_banda = ['#4CAF50', '#8BC34A', '#FFC107', '#FF9800', '#F44336']
bandas = resumen_bandas.index.tolist()

# 1. Distribucion de clientes
axes[0].bar(range(len(bandas)), resumen_bandas['pct_total'],
            color=colores_banda, edgecolor='white')
axes[0].set_xticks(range(len(bandas)))
axes[0].set_xticklabels([b.split(' - ')[0] for b in bandas], fontsize=11)
axes[0].set_ylabel('% del total')
axes[0].set_title('Distribucion de clientes')
for i, v in enumerate(resumen_bandas['pct_total']):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# 2. Tasa de incumplimiento por banda
axes[1].bar(range(len(bandas)), resumen_bandas['tasa_default'],
            color=colores_banda, edgecolor='white')
axes[1].set_xticks(range(len(bandas)))
axes[1].set_xticklabels([b.split(' - ')[0] for b in bandas], fontsize=11)
axes[1].set_ylabel('Tasa de default (%)')
axes[1].set_title('Tasa de incumplimiento')
for i, v in enumerate(resumen_bandas['tasa_default']):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# 3. Score medio por banda
axes[2].bar(range(len(bandas)), resumen_bandas['score_medio'],
            color=colores_banda, edgecolor='white')
axes[2].set_xticks(range(len(bandas)))
axes[2].set_xticklabels([b.split(' - ')[0] for b in bandas], fontsize=11)
axes[2].set_ylabel('Score medio')
axes[2].set_title('Score medio por banda')
for i, v in enumerate(resumen_bandas['score_medio']):
    axes[2].text(i, v + 2, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '16_bandas_de_riesgo.png'), bbox_inches='tight')
plt.show()

## 8. Caso de uso: perfil de ejemplo

Para ilustrar el uso práctico de la scorecard, se muestran ejemplos de cómo 
un individuo puede conocer su score y cómo se compara con la población.

In [ ]:
# Cuantiles de referencia para comparacion
cuantiles = [10, 25, 50, 75, 90]
scores_cuantiles = np.percentile(df_probas['score'], cuantiles)

print('Percentiles de la distribucion de scores:')
for q, s in zip(cuantiles, scores_cuantiles):
    p = score_a_probabilidad(s)
    print(f'  Percentil {q:3d}: Score = {s:.0f} | Prob. default = {p*100:.2f}%')

In [ ]:
# Funcion para evaluar un perfil
def evaluar_perfil(score_individuo, df_scores, nombre='Individuo'):
    """
    Muestra el perfil de riesgo de un individuo y como se compara con la poblacion.
    """
    prob = score_a_probabilidad(score_individuo)
    banda = asignar_banda(score_individuo)
    percentil = (df_scores['score'] <= score_individuo).mean() * 100
    
    print(f'--- Perfil de {nombre} ---')
    print(f'  Score                    : {score_individuo}')
    print(f'  Probabilidad de default  : {prob*100:.2f}%')
    print(f'  Banda de riesgo          : {banda}')
    print(f'  Percentil en la poblacion: {percentil:.1f}%')
    print(f'  (Mejor que el {percentil:.0f}% de los solicitantes)\n')
    
    # Grafica
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_scores['score'], bins=50, color='#E0E0E0', edgecolor='white',
            density=True, label='Poblacion')
    ax.axvline(score_individuo, color='#F44336', linewidth=3,
               label=f'{nombre}: {score_individuo} pts')
    ax.set_xlabel('Score')
    ax.set_ylabel('Densidad')
    ax.set_title(f'Posicion de {nombre} en la distribucion de scores',
                 fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()


# Ejemplo 1: persona de bajo riesgo
evaluar_perfil(720, df_probas, 'Persona A (bajo riesgo)')

# Ejemplo 2: persona de riesgo medio
evaluar_perfil(590, df_probas, 'Persona B (riesgo medio)')

# Ejemplo 3: persona de alto riesgo
evaluar_perfil(480, df_probas, 'Persona C (alto riesgo)')

## 9. Guardado de la scorecard

In [ ]:
# Guardado del dataset con scores
df_probas.to_csv(os.path.join(OUTPUT_PATH, 'scorecard_test.csv'), index=False)

# Guardado de la tabla de factores de riesgo
df_factores.to_csv(os.path.join(OUTPUT_PATH, 'factores_riesgo.csv'), index=False)

# Guardado de las bandas de riesgo
resumen_bandas.to_csv(os.path.join(OUTPUT_PATH, 'bandas_riesgo.csv'))

# Guardado de los bins WOE (necesarios para la app web)
joblib.dump(bins, os.path.join(OUTPUT_PATH, 'woe_bins.pkl'))

# Guardar parametros de la scorecard
params_scorecard = {
    'score_base': SCORE_BASE,
    'odds_base': ODDS_BASE,
    'pdo': PDO,
    'factor': FACTOR,
    'offset': OFFSET
}
joblib.dump(params_scorecard, os.path.join(OUTPUT_PATH, 'params_scorecard.pkl'))

print('Archivos guardados:')
print(f'  scorecard_test.csv    -> {OUTPUT_PATH}')
print(f'  factores_riesgo.csv   -> {OUTPUT_PATH}')
print(f'  bandas_riesgo.csv     -> {OUTPUT_PATH}')
print(f'  woe_bins.pkl          -> {OUTPUT_PATH}')
print(f'  params_scorecard.pkl  -> {OUTPUT_PATH}')

## Resumen

### Pasos realizados

1. **Construcción de la scorecard:** se convirtieron las probabilidades de incumplimiento
   del modelo neuronal en un puntaje entero (300-850 pts) usando la transformación estándar
   log-odds con PDO = 20 y score base = 600.
2. **Verificación de calibración:** se confirmó que la relación score-probabilidad es
   monótona decreciente y coherente con los parámetros configurados.
3. **Distribución de scores:** los buenos pagadores presentan scores significativamente
   mayores que los malos pagadores, confirmando el poder discriminatorio del modelo.
4. **Análisis de variables de riesgo:** mediante IV y WOE se identificaron las variables
   que más influyen en el riesgo, junto con las categorías específicas asociadas a mayor
   probabilidad de incumplimiento.
5. **Segmentación:** se crearon 5 bandas de riesgo (A-E) con tasas de incumplimiento
   progresivamente mayores, útiles para la toma de decisiones de crédito.
6. **Caso de uso:** se demostró cómo un individuo puede conocer su score y compararse
   con la población, funcionalidad que se implementará en la aplicación web.

### Hallazgos clave sobre variables de riesgo

- Las variables con mayor poder predictivo tienden a ser las relacionadas con el historial
  crediticio del solicitante (grade, sub_grade) y las condiciones del préstamo (int_rate, term).
- Los perfiles de mayor riesgo se caracterizan por: tasas de interés altas, grados de riesgo
  bajos (E, F, G), historial crediticio corto, y altos niveles de endeudamiento.
- La scorecard resultante es interpretable: cada 20 puntos adicionales de score corresponden
  a una reducción a la mitad en la probabilidad de incumplimiento.

### Siguiente paso — Aplicación web

En la siguiente fase se implementará una aplicación web (Streamlit) que permita a los
usuarios ingresar sus características y obtener su score crediticio, junto con una
visualización de cómo se posicionan respecto a la población.